<a href="https://colab.research.google.com/github/MoEissa140/FlyRank-Intern/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MoEissa140/FlyRank-Intern/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [ ]:
!pip install -q huggingface_hub duckdb
import duckdb, os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
import pandas as pd

In [ ]:
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
CREATE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{os.environ["HF_TOKEN"]}'
);
""")

base = "hf://datasets/FlyRank/internship-warehouse"

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=os.environ["HF_TOKEN"])
for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [ ]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{base}/dim_content.parquet') LIMIT 1").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [ ]:
con.sql(f"""
SELECT
  approx_quantile(gsc_impressions, [0.1,0.5,0.9,0.99]) AS impressions_q,
  approx_quantile(gsc_avg_position, [0.1,0.5,0.9,0.99]) AS position_q,
  approx_quantile(gsc_clicks * 1.0 / NULLIF(gsc_impressions,0), [0.1,0.5,0.9,0.99]) AS ctr_q
FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_impressions > 0
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,impressions_q,position_q,ctr_q
0,"[1, 16, 185, 949]","[1.4787947163986208, 7.477516043711573, 43.125...","[0.0, 0.0, 0.0030245483651393464, 0.0507273704..."


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [ ]:
# Test 1 (flag-linked): CTR vs position tier — underlies CTR-fix logic
con.sql(f"""
SELECT
  CASE WHEN gsc_avg_position <= 3 THEN 'top3'
       WHEN gsc_avg_position <= 10 THEN 'page1'
       WHEN gsc_avg_position <= 20 THEN 'page2'
       ELSE 'beyond' END AS position_tier,
  AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions,0)) AS avg_ctr,
  COUNT(*) AS n
FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_impressions > 0
GROUP BY 1 ORDER BY 2 DESC
""").df()
# Verdict: CONFIRMED if avg_ctr strictly decreases as tier worsens

# Test 2: volume vs demand (quick-win logic)
con.sql(f"""
SELECT
  CASE WHEN gsc_impressions >= 500 THEN 'high_volume' ELSE 'low_volume' END AS volume_bucket,
  AVG(gsc_avg_position) AS avg_position,
  COUNT(*) AS n
FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY 1
""").df()
# Verdict: label MIXED/CONFIRMED based on whether high-volume pages skew toward better position

# Test 3: engagement vs sessions volume
con.sql(f"""
SELECT
  SUM(ga4_sessions) AS total_sessions,
  SUM(ga4_engaged_sessions) AS total_engaged,
  MAX(ga4_engaged_sessions) AS max_engaged_in_a_row,
  MAX(ga4_sessions) AS max_sessions_in_a_row
FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE ga4_data_available IS TRUE
""").df()
# Verdict based on printed numbers

,total_sessions,total_engaged,max_engaged_in_a_row,max_sessions_in_a_row
0,1299808.0,29551.0,21,792


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [ ]:
con.sql(f"""
SELECT
  CASE WHEN gsc_avg_position <= 20 AND gsc_impressions >= 500 THEN 'flag_eligible' ELSE 'not_eligible' END AS flag_bucket,
  AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions,0)) * 100 AS avg_ctr_pct,
  COUNT(*) AS n
FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_impressions > 0
GROUP BY 1
""").df()

,flag_bucket,avg_ctr_pct,n
0,not_eligible,0.307584,3533503
1,flag_eligible,0.330441,77558


FlyRank's `low_ctr_visible_page` flag assumes: pages with real visibility (≥500 impressions) and a reasonable position (≤20) but CTR under 0.5% deserve a metadata/title review. Checking the assumption underneath — do "flag_eligible" pages (high impressions, good position) actually get meaningfully more clicks per impression than pages that don't meet that bar?

| flag_bucket | avg_ctr_pct | n |
|---|---|---|
| flag_eligible | [0.307584] | [3533503] |
| not_eligible | [0.330441] | [77558] |

**Verdict:** [CONFIRMED if flag_eligible's CTR is clearly higher than not_eligible's, meaning the flag's implicit assumption (visible+positioned pages should get proportionally more clicks) holds / MIXED or OPPOSITE if not — say so plainly either way].

## 4. What this means in practice

If CTR-vs-position is CONFIRMED: a content team can trust position as a proxy signal for CTR review priority without needing engagement data (which is only ~4% available). If MIXED or FALSE: flag that the existing CTR-fix rule may be over- or under-triggering, and recommend combining position with a minimum-impression filter before trusting it.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.